<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    X, y = fetch_openml("mnist_784", as_frame=False, return_X_y=True)

    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_test, y_train, y_test


# Para Random Forest e AdaBoost com árvores, não é necessário normalizar os dados.
# Esses modelos trabalham com divisões por limiar nas features, então a escala dos 
# valores não costuma afetar o aprendizado como afeta em modelos baseados em distância 
# ou gradiente.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [16]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [12]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    return acc

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [7]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    acc = evaluate(model, X_test, y_test)
    
    return acc

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [10]:
depths = range(1, 21)

X_train, X_test, y_train, y_test = load_data(42)

print(f"{'Depth':<10}{'Train Acc':<15}{'Test Acc':<15}")

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    print(f"{d:<10}{train_acc:<15.4f}{test_acc:<15.4f}")

Depth     Train Acc      Test Acc       
1         0.1989         0.1982         
2         0.3421         0.3434         
3         0.4422         0.4444         
4         0.5619         0.5598         
5         0.6591         0.6554         
6         0.7319         0.7291         
7         0.7851         0.7789         
8         0.8277         0.8144         
9         0.8693         0.8414         
10        0.9057         0.8613         
11        0.9330         0.8713         
12        0.9538         0.8760         
13        0.9693         0.8781         
14        0.9799         0.8799         
15        0.9868         0.8812         
16        0.9900         0.8776         
17        0.9923         0.8758         
18        0.9935         0.8772         
19        0.9944         0.8786         
20        0.9951         0.8768         


Max_depth = 15 é o ponto máximo da acurácia de teste.Então o overfitting começa a partir de max_depth = 16.

Porque a árvore tenta o grau máximo de pureza e se não tiver limite, ela vai criar ramos até que a pureza seja máxima, independente de quantas decisões sejam criadas.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [17]:
def evaluate_full(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    return acc, precision, recall, f1

In [18]:
X_train, X_test, y_train, y_test = load_data(42)

rf = train_random_forest(X_train, y_train, 42)
ab = train_adaboost(X_train, y_train, 42)

rf_metrics = evaluate_full(rf, X_test, y_test)
ab_metrics = evaluate_full(ab, X_test, y_test)

print("Random Forest")
print(f"Acurácia: {rf_metrics[0]:.4f}")
print(f"Precisão: {rf_metrics[1]:.4f}")
print(f"Recall: {rf_metrics[2]:.4f}")
print(f"F1-score: {rf_metrics[3]:.4f}")

print("\nAdaBoost")
print(f"Acurácia: {ab_metrics[0]:.4f}")
print(f"Precisão: {ab_metrics[1]:.4f}")
print(f"Recall: {ab_metrics[2]:.4f}")
print(f"F1-score: {ab_metrics[3]:.4f}")

Random Forest
Acurácia: 0.9672
Precisão: 0.9672
Recall: 0.9672
F1-score: 0.9672

AdaBoost
Acurácia: 0.6681
Precisão: 0.6884
Recall: 0.6681
F1-score: 0.6711


O modelo Random Forest apresentou melhor desempenho inicial, com acurácia, precisão, recall e F1-score significativamente superiores ao AdaBoost. Enquanto o Random Forest obteve aproximadamente 96,7% em todas as métricas, o AdaBoost apresentou desempenho inferior, com cerca de 66–68%.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
# TODO: implemente load_data

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
# TODO: implemente load_data

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
# TODO: implemente load_data

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
# TODO: implemente load_data